In [38]:
import time
import requests
import json
from pathlib import Path
import pandas as pd
from data import load_data

data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\diagnoses_per_stays.csv"
dictionary_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\diagnoses_dictionary.csv"
STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\diagnosess_snomed_state.json")
concept_path = r"D:\Python Projects\Hospital readmission risk\data\concepts"
write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\main_diagnoses.csv"
CACHE: dict[str, str] = {}
sql = """
select distinct
i.stay_id,
  cl.diagnosis1,
  cl.diagnosis2,
  cl.diagnosis3,
  cl.diagnosis4,
  cl.diagnosis5,
  cl.diagnosis6,
  cl.diagnosis7,
  cl.diagnosis8
from hospital-readmission-4.helper_tables.index_stay i
left join hospital-readmission-4.data_slim.claims_slim cl
on i.stay_id = cl.encounter
"""
output_cols = ['main_diagnosis_code', 'main_diagnosis_name', 'main_diagnosis_type',
'num_of_disorders', 'num_of_findings']
viral_codes = {'34014006'}

In [2]:
def get_dictionary(path):

    dictionary = pd.read_csv(path)

    dictionary.set_index('code', inplace = True)

    return dictionary

In [3]:
def disorders_and_symptoms_split(codes: set, dictionary):

    disorders = set()
    symptoms = set()

    for code in codes:

        if int(code) in dictionary.index:

            if dictionary.loc[int(code), 'is_disorder'] == 1:
                disorders.add(code)

            if dictionary.loc[int(code), 'is_symptom'] == 1:
                symptoms.add(code)

    return disorders, symptoms


In [4]:
def load_state():

    global CACHE

    if not STATE_PATH.exists():
        return

    with STATE_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    raw_cache = data.get("cache", {})
    CACHE = {code: path for code, path in raw_cache.items()}

def save_concept(concept_id, data, save_path = concept_path):

    path = Path(save_path)
    path.mkdir(parents=True, exist_ok=True)

    file_name = f"{concept_id}.json"
    file_path = path/file_name

    with file_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    CACHE[concept_id] = str(file_path)

def read_concept(concept_path):

    path = Path(concept_path)

    if not path.exists():
        return {}
    
    with path.open("r", encoding="utf-8") as f:
        return json.load(f) 

def get_concept(concept_id: str) -> dict:

    if concept_id in CACHE:

        return read_concept(CACHE[concept_id])

def get_parent_ids(concept: dict) -> list[str]:

    """Extract parent conceptIds from the concept JSON (adjust to actual structure)."""

    parents = []

    for rel in concept.get("relationships", []):
        # 'is a' relationship has conceptId 116680003 in SNOMED CT
        if rel.get("type", {}).get("conceptId") == "116680003" and rel.get("active"):

            target = rel.get("target") or {}

            cid = target.get("conceptId")

            if cid:
                
                parents.append(cid)
            
    return parents

def is_or_has_ancestor_in(concept_id: str, target_ids: set[str], max_depth: int = 10) -> bool:
    # if we’ve already fully explored this concept’s ancestors in a previous run,
    # just return the previous answer if you also store it, or skip re-walk.
    visited = set()
    frontier = {concept_id}
    result = False

    if concept_id in target_ids:
        return True

    for depth in range(max_depth):

        next_frontier = set()

        for cid in frontier:

            if cid in visited:
                continue

            visited.add(cid)

            c = get_concept(cid)

            if not isinstance(c, dict):
                continue

            parents = get_parent_ids(c)

            if target_ids.intersection(parents):
                return True
            
            next_frontier.update(p for p in parents if p not in visited)

        if not next_frontier:
            break
        
        frontier = next_frontier

    return result

In [5]:
def get_description(concept_id: str):

    if CACHE[concept_id]:

        return get_concept(concept_id)["descriptions"][0]["term"]

    return None

In [32]:
def find_main_disorder(disorders: set) -> str:

    if len(disorders) == 1:
        return disorders.pop()

    for code in disorders:

        if is_or_has_ancestor_in(code, viral_codes):
            return code

    suspects = disorders.copy()
    visited = set()
    frontier = set(disorders)

    for depth in range(20):

        next_frontier = frontier.copy()

        for cid in frontier:
            
            if cid in visited: 
                continue

            visited.add(cid)

            c = get_concept(cid)

            if not isinstance(c, dict):
                continue

            parents = get_parent_ids(c)
            
            if frontier.intersection(set(parents)) or suspects.intersection(set(parents)):

                intersect = frontier.intersection(set(parents))
                intersect.update(suspects.intersection(set(parents)))

                for disorder in list(suspects):

                    if is_or_has_ancestor_in(disorder, intersect, depth):

                        suspects.discard(disorder)

                        if len(suspects) == 1:
                            return suspects.pop()
            next_frontier.update(p for p in parents if p not in visited)

        frontier = next_frontier
            
    print("Returned minimum, initial set:" + str(disorders) + ", suspects: " + str(suspects))
    return suspects.pop()
    

In [ ]:
def get_codes(data, id) -> set(str()):

    codes = set ()

    cols = data.loc[id, 'diagnosis1':'diagnosis8'].dropna()

    for col in cols:

        codes.add(str(int(col)))

    return codes


In [30]:
def build_main_diagnoses(data):

    data.set_index('stay_id', inplace = True)

    main_diagnoses = pd.DataFrame(index = data.index, columns = output_cols)

    dictionary = get_dictionary(dictionary_path)

    for id in data.index:

        codes = get_codes(data, id)
        
        disorders, symptoms = disorders_and_symptoms_split(codes, dictionary)

        main_diagnoses.loc[id, 'num_of_disorders'] = len(disorders)
        main_diagnoses.loc[id, 'num_of_findings'] = len(symptoms)

        if len(disorders) == 0:
            
            if len(symptoms) > 0:
                
                min_symptom = symptoms.pop()

                main_diagnoses.loc[id, 'main_diagnosis_code'] = min_symptom
                main_diagnoses.loc[id, 'main_diagnosis_type'] = 'finding'

        else: 

            main_diagnoses.loc[id, 'main_diagnosis_type'] = 'disorder'
            main_diagnoses.loc[id, 'main_diagnosis_code'] = find_main_disorder(disorders)

        if not pd.isna(main_diagnoses.loc[id, 'main_diagnosis_code']):
            main_diagnoses.loc[id, 'main_diagnosis_name'] = get_description(main_diagnoses.loc[id, 'main_diagnosis_code'])

    return main_diagnoses

In [39]:
def pack_dictionary(data, path: str):

    data.to_csv(path)

In [33]:
data = load_data(data_path, sql)
load_state()
main_diagnoses = build_main_diagnoses(data)

In [40]:
pack_dictionary(main_diagnoses, write_path)

---

In [25]:
ids = ['ada84aac-5385-111f-35f6-d912d5541d00', '17a2ea60-f01d-a6bd-cdf6-5b7f22cf0324']
data.loc[ids]
test_diag = build_main_diagnoses(data.loc[ids])

cid: 237602007
parents: ['75934005']
suspects: {'237602007', '80394007', '302870006'}
frontier: {'237602007', '80394007', '302870006'}
nest_frontier: {'237602007', '80394007', '302870006', '75934005'}
237602007: failed
cid: 80394007
parents: ['126877002']
suspects: {'237602007', '80394007', '302870006'}
frontier: {'237602007', '80394007', '302870006'}
nest_frontier: {'237602007', '126877002', '80394007', '75934005', '302870006'}
80394007: failed
cid: 302870006
parents: ['55822004']
suspects: {'237602007', '80394007', '302870006'}
frontier: {'237602007', '80394007', '302870006'}
nest_frontier: {'237602007', '126877002', '80394007', '75934005', '55822004', '302870006'}
302870006: failed
cid: 126877002
parents: ['20957000']
suspects: {'237602007', '80394007', '302870006'}
frontier: {'237602007', '126877002', '80394007', '75934005', '55822004', '302870006'}
nest_frontier: {'237602007', '55822004', '126877002', '302870006', '20957000', '80394007', '75934005'}
126877002: failed
cid: 75934005

In [26]:
test_diag

,main_diagnosis_code,main_diagnosis_name,main_diagnosis_type,num_of_disorders,num_of_findings
stay_id,,,,,
ada84aac-5385-111f-35f6-d912d5541d00,302870006,Hypertriglyceridaemia,disorder,3,1
17a2ea60-f01d-a6bd-cdf6-5b7f22cf0324,302870006,Hypertriglyceridaemia,disorder,5,2


In [80]:
test_diag.loc['bc547828-5aab-faa1-0fa3-3977e9a3d769','main_diagnosis_name'] = get_description(test_diag.loc['bc547828-5aab-faa1-0fa3-3977e9a3d769','main_diagnosis_code'])

In [11]:
data[data['diagnosis1'] == 80394007]

,diagnosis1,diagnosis2,diagnosis3,diagnosis4,diagnosis5,diagnosis6,diagnosis7,diagnosis8
stay_id,,,,,,,,
ada84aac-5385-111f-35f6-d912d5541d00,80394007,302870006.0,237602007.0,160903007.0,NaN,NaN,NaN,NaN
cff7375d-1624-3ad1-6e9d-0f2fb12ca382,80394007,73438004.0,NaN,NaN,NaN,NaN,NaN,NaN
36c86f70-85b0-1d4b-8598-c73c3765575e,80394007,302870006.0,237602007.0,160904001.0,NaN,NaN,NaN,NaN
5a0b30ef-178a-98e6-7a31-a328bbaf1814,80394007,302870006.0,237602007.0,424393004.0,NaN,NaN,NaN,NaN
59bcac56-a975-109e-808f-11351554f8cf,80394007,160904001.0,422650009.0,NaN,NaN,NaN,NaN,NaN
17a2ea60-f01d-a6bd-cdf6-5b7f22cf0324,80394007,302870006.0,237602007.0,271737000.0,73595000.0,424393004.0,66383009.0,NaN


In [20]:
for depth in range (0):
    print(depth)